# Hackathon AI & ML — Advanced v3
**Base** : même pipeline que `baseline.ipynb`  
**Améliorations vs baseline** :
- Seuil NaN à 70% → plus de features
- Features NaN par composante (labo / exam / questionnaire)
- **Features cliniques** : moyennes et taux de manquants par groupe fonctionnel (vitalité, mobilité, cognition, sensoriel, psychologique)
- `scale_pos_weight` pour le déséquilibre de classes
- **Meilleurs hyperparamètres LightGBM** issus d'Optuna (50 trials)
- **Ensemble 4 modèles** : LightGBM + XGBoost + CatBoost + Random Forest (OOF)
- **SMOTE par fold** pour Random Forest
- **Poids d'ensemble optimisés** par Nelder-Mead sur OOF
- **Stacking** LogisticRegression + calibration isotonique
- **Threshold optimal** sur OOF
- Sélection automatique de la meilleure stratégie

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import minimize

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

DATA_DIR          = Path('data')
GROUP_ID          = 'G1'
SUBMISSION_ID     = '4'
RANDOM_STATE      = 42
N_FOLDS           = 5
MISSING_THRESHOLD = 0.70

print('Librairies chargées OK')

## 1. Chargement des données

In [ ]:
print('Chargement des données...')
data     = pd.read_csv(DATA_DIR / 'data.csv')
labels   = pd.read_csv(DATA_DIR / 'ground_truth_train.csv')
test_idx = pd.read_csv(DATA_DIR / 'test_indexes.csv', header=None, names=['SEQN'])
metadata = pd.read_csv(DATA_DIR / 'features_metadata.csv')

print(f'data.csv        : {data.shape}')
print(f'ground_truth    : {labels.shape}')
print(f'test_indexes    : {test_idx.shape}')
print(f'features_meta   : {metadata.shape}')

test_seqn  = set(test_idx['SEQN'].values)
train_seqn = set(labels['SEQN'].values) - test_seqn

train_data = data[data['SEQN'].isin(train_seqn)].copy()
test_data  = data[data['SEQN'].isin(test_seqn)].copy()
train_df   = train_data.merge(labels[labels['SEQN'].isin(train_seqn)], on='SEQN')

print(f'\nTrain : {train_df.shape}  |  Test : {test_data.shape}')
print(f'Taux de mortalité : {train_df["MORTSTAT_2019"].mean():.2%}')

## 2. Feature engineering

In [ ]:
# --- Colonnes par composante ---
meta_cols  = metadata[['SAS', 'Component']].rename(columns={'SAS': 'col'})
labo_cols  = meta_cols[meta_cols['Component'] == 'Laboratory']['col'].tolist()
exam_cols  = meta_cols[meta_cols['Component'] == 'Examination']['col'].tolist()
quest_cols = meta_cols[meta_cols['Component'] == 'Questionnaire']['col'].tolist()

# --- Colonnes par groupe fonctionnel clinique ---
def get_func_cols(metadata, keyword):
    return metadata[metadata['function'].apply(lambda f: keyword in str(f))]['SAS'].tolist()

func_groups = {
    'vitality':  get_func_cols(metadata, 'Vitalité'),
    'mobility':  get_func_cols(metadata, 'Mobilité'),
    'cognition': get_func_cols(metadata, 'Cognition'),
    'sensory':   get_func_cols(metadata, 'Sensoriel'),
    'psycho':    get_func_cols(metadata, 'Psychologique'),
}
for k, v in func_groups.items():
    print(f'  {k}: {len(v)} variables')

def add_features(df, labo_cols, exam_cols, quest_cols, func_groups):
    all_cols = [c for c in df.columns if c not in ['SEQN', 'MORTSTAT_2019']]
    df = df.copy()

    # NaN par composante
    df['feat_nb_missing_total'] = df[all_cols].isnull().sum(axis=1)
    df['feat_pct_missing_total'] = df['feat_nb_missing_total'] / len(all_cols)
    for name, cols in [('labo', labo_cols), ('exam', exam_cols), ('quest', quest_cols)]:
        present = [c for c in cols if c in df.columns]
        if present:
            df[f'feat_nb_missing_{name}'] = df[present].isnull().sum(axis=1)

    # Agrégats par groupe fonctionnel clinique
    for name, cols in func_groups.items():
        present = [c for c in cols if c in df.columns]
        if present:
            df[f'feat_{name}_mean']    = df[present].mean(axis=1)
            df[f'feat_{name}_missing'] = df[present].isnull().mean(axis=1)
    return df

train_df  = add_features(train_df,  labo_cols, exam_cols, quest_cols, func_groups)
test_data = add_features(test_data, labo_cols, exam_cols, quest_cols, func_groups)
print(f'\nFeatures engineering terminé. Shape train : {train_df.shape}')

## 3. Nettoyage et préparation

In [ ]:
missing_ratio = train_df.isnull().mean()
cols_to_drop  = [c for c in missing_ratio[missing_ratio > MISSING_THRESHOLD].index
                 if c not in ['SEQN', 'MORTSTAT_2019']]

train_clean = train_df.drop(columns=cols_to_drop)
test_clean  = test_data.drop(columns=[c for c in cols_to_drop if c in test_data.columns])

feature_cols    = [c for c in train_clean.columns if c not in ['SEQN', 'MORTSTAT_2019']]
X               = train_clean[feature_cols]
y               = train_clean['MORTSTAT_2019']
X_test          = test_clean[feature_cols]
test_seqn_order = test_clean['SEQN']

neg_pos_ratio = (y == 0).sum() / (y == 1).sum()

print(f'Colonnes supprimées (>{MISSING_THRESHOLD*100:.0f}% manquant) : {len(cols_to_drop)}')
print(f'Features finales    : {X.shape[1]}')
print(f'Exemples train      : {X.shape[0]}')
print(f'Exemples test       : {X_test.shape[0]}')
print(f'Déséquilibre (0/1)  : {neg_pos_ratio:.1f}x')

## 4. Paramètres des modèles

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# LightGBM — meilleurs params Optuna (50 trials)
lgb_params = {
    'objective':         'binary',
    'metric':            'binary_logloss',
    'verbose':           -1,
    'n_jobs':            -1,
    'random_state':      RANDOM_STATE,
    'scale_pos_weight':  neg_pos_ratio,
    'n_estimators':      2000,
    'learning_rate':     0.023688639503640783,
    'num_leaves':        244,
    'min_child_samples': 76,
    'subsample':         0.7993292420985183,
    'colsample_bytree':  0.5780093202212182,
    'reg_alpha':         0.004207053950287938,
    'reg_lambda':        0.0017073967431528124,
}

# XGBoost
xgb_params = {
    'n_estimators':      2000,
    'learning_rate':     0.05,
    'max_depth':         6,
    'min_child_weight':  10,
    'subsample':         0.8,
    'colsample_bytree':  0.6,
    'reg_alpha':         0.01,
    'reg_lambda':        1.0,
    'scale_pos_weight':  neg_pos_ratio,
    'random_state':      RANDOM_STATE,
    'n_jobs':            -1,
    'verbosity':         0,
    'eval_metric':       'logloss',
}

# CatBoost
cat_params = {
    'iterations':            1000,
    'learning_rate':         0.05,
    'depth':                 6,
    'loss_function':         'Logloss',
    'eval_metric':           'F1',
    'class_weights':         [1, neg_pos_ratio],
    'random_seed':           RANDOM_STATE,
    'verbose':               0,
    'early_stopping_rounds': 50,
}

# Random Forest
rf_params = {
    'n_estimators':     500,
    'max_depth':        20,
    'min_samples_leaf': 10,
    'max_features':     'sqrt',
    'class_weight':     'balanced',
    'random_state':     RANDOM_STATE,
    'n_jobs':           -1,
}

print('Paramètres initialisés (LGB, XGB, CatBoost, RF)')

## 5. Entraînement CV — OOF predictions

In [ ]:
# Initialisation des tableaux OOF
oof_lgb  = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))
oof_rf   = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))
test_rf  = np.zeros(len(X_test))

# Imputation médiane pour XGB et RF (ne gèrent pas les NaN nativement)
train_medians = X.median()
X_imp      = X.fillna(train_medians)
X_test_imp = X_test.fillna(train_medians)

print('Tableaux OOF initialisés')

In [ ]:
# --- LightGBM ---
print('=== LightGBM (params Optuna) ===')
f1_lgb = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(X_tr, y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(period=-1)])
    oof_lgb[val_idx]  = m.predict_proba(X_val)[:, 1]
    test_lgb         += m.predict_proba(X_test)[:, 1] / N_FOLDS
    f1 = f1_score(y_val, (oof_lgb[val_idx] > 0.5).astype(int))
    f1_lgb.append(f1)
    print(f'  Fold {fold+1} F1: {f1:.4f}')
print(f'  LGB CV F1: {np.mean(f1_lgb):.4f} ± {np.std(f1_lgb):.4f}')

In [ ]:
# --- XGBoost ---
print('=== XGBoost ===')
f1_xgb = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y)):
    X_tr, X_val = X_imp.iloc[tr_idx], X_imp.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    m = xgb.XGBClassifier(**xgb_params)
    m.fit(X_tr, y_tr,
          eval_set=[(X_val, y_val)],
          early_stopping_rounds=50,
          verbose=False)
    oof_xgb[val_idx]  = m.predict_proba(X_val)[:, 1]
    test_xgb         += m.predict_proba(X_test_imp)[:, 1] / N_FOLDS
    f1 = f1_score(y_val, (oof_xgb[val_idx] > 0.5).astype(int))
    f1_xgb.append(f1)
    print(f'  Fold {fold+1} F1: {f1:.4f}')
print(f'  XGB CV F1: {np.mean(f1_xgb):.4f} ± {np.std(f1_xgb):.4f}')

In [ ]:
# --- CatBoost ---
print('=== CatBoost ===')
f1_cat = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    m = CatBoostClassifier(**cat_params)
    m.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    oof_cat[val_idx]  = m.predict_proba(X_val)[:, 1]
    test_cat         += m.predict_proba(X_test)[:, 1] / N_FOLDS
    f1 = f1_score(y_val, (oof_cat[val_idx] > 0.5).astype(int))
    f1_cat.append(f1)
    print(f'  Fold {fold+1} F1: {f1:.4f}')
print(f'  CAT CV F1: {np.mean(f1_cat):.4f} ± {np.std(f1_cat):.4f}')

In [ ]:
# --- Random Forest + SMOTE par fold ---
print('=== Random Forest + SMOTE ===')
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
f1_rf = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y)):
    X_tr, X_val = X_imp.iloc[tr_idx], X_imp.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    # SMOTE uniquement sur le fold d'entraînement (pas de fuite)
    X_tr_s, y_tr_s = smote.fit_resample(X_tr, y_tr)
    m = RandomForestClassifier(**rf_params)
    m.fit(X_tr_s, y_tr_s)
    oof_rf[val_idx]  = m.predict_proba(X_val)[:, 1]
    test_rf         += m.predict_proba(X_test_imp)[:, 1] / N_FOLDS
    f1 = f1_score(y_val, (oof_rf[val_idx] > 0.5).astype(int))
    f1_rf.append(f1)
    print(f'  Fold {fold+1} F1: {f1:.4f}')
print(f'  RF  CV F1: {np.mean(f1_rf):.4f} ± {np.std(f1_rf):.4f}')

## 6. Ensemble — poids optimisés + stacking + calibration

In [ ]:
# --- A. Averaging avec poids optimisés par Nelder-Mead (4 modèles) ---
def neg_f1_weighted(weights, threshold=0.5):
    w = np.abs(weights) / np.abs(weights).sum()
    oof = w[0]*oof_lgb + w[1]*oof_xgb + w[2]*oof_cat + w[3]*oof_rf
    return -f1_score(y, (oof > threshold).astype(int))

res = minimize(neg_f1_weighted, [0.25, 0.25, 0.25, 0.25], method='Nelder-Mead',
               options={'maxiter': 2000, 'xatol': 1e-5})
opt_w = np.abs(res.x) / np.abs(res.x).sum()

oof_avg  = opt_w[0]*oof_lgb  + opt_w[1]*oof_xgb  + opt_w[2]*oof_cat  + opt_w[3]*oof_rf
test_avg = opt_w[0]*test_lgb + opt_w[1]*test_xgb + opt_w[2]*test_cat + opt_w[3]*test_rf

thresholds   = np.arange(0.20, 0.70, 0.01)
f1_avg_thr   = [f1_score(y, (oof_avg > t).astype(int)) for t in thresholds]
best_thr_avg = thresholds[np.argmax(f1_avg_thr)]
best_f1_avg  = max(f1_avg_thr)

print(f'Poids optimisés : LGB={opt_w[0]:.3f}  XGB={opt_w[1]:.3f}  CAT={opt_w[2]:.3f}  RF={opt_w[3]:.3f}')
print(f'Averaging OOF F1 (thr={best_thr_avg:.2f}) : {best_f1_avg:.4f}')

In [ ]:
# --- B. Stacking + calibration isotonique ---
meta_X_train = np.column_stack([oof_lgb, oof_xgb, oof_cat, oof_rf])
meta_X_test  = np.column_stack([test_lgb, test_xgb, test_cat, test_rf])

scaler = StandardScaler()
meta_X_train_s = scaler.fit_transform(meta_X_train)
meta_X_test_s  = scaler.transform(meta_X_test)

meta = LogisticRegression(C=1.0, random_state=RANDOM_STATE, max_iter=1000)
meta.fit(meta_X_train_s, y)

oof_stack_raw  = meta.predict_proba(meta_X_train_s)[:, 1]
test_stack_raw = meta.predict_proba(meta_X_test_s)[:, 1]

# Calibration isotonique — correction des probabilités sur OOF
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(oof_stack_raw, y)
oof_stack  = iso.transform(oof_stack_raw)
test_stack = iso.transform(test_stack_raw)

f1_stack_thr   = [f1_score(y, (oof_stack > t).astype(int)) for t in thresholds]
best_thr_stack = thresholds[np.argmax(f1_stack_thr)]
best_f1_stack  = max(f1_stack_thr)

print(f'Stacking OOF F1 (thr={best_thr_stack:.2f}) : {best_f1_stack:.4f}')

In [ ]:
# --- Comparaison globale ---
print('=== Comparaison OOF F1 ===')
print(f'  Baseline LGB (seuil 0.5)   : 0.6945')
print(f'  Baseline + spw + threshold : 0.7023')
print(f'  LGB Optuna                 : {np.mean(f1_lgb):.4f} ± {np.std(f1_lgb):.4f}')
print(f'  XGBoost                    : {np.mean(f1_xgb):.4f} ± {np.std(f1_xgb):.4f}')
print(f'  CatBoost                   : {np.mean(f1_cat):.4f} ± {np.std(f1_cat):.4f}')
print(f'  Random Forest + SMOTE      : {np.mean(f1_rf):.4f}  ± {np.std(f1_rf):.4f}')
print(f'  Averaging optimisé         : {best_f1_avg:.4f}  (thr={best_thr_avg:.2f})')
print(f'  Stacking + calibration     : {best_f1_stack:.4f}  (thr={best_thr_stack:.2f})')

# Courbes threshold
plt.figure(figsize=(9, 4))
plt.plot(thresholds, f1_avg_thr,   label=f'Averaging  (best={best_f1_avg:.4f})')
plt.plot(thresholds, f1_stack_thr, label=f'Stacking+calib (best={best_f1_stack:.4f})')
plt.axvline(best_thr_avg,   color='C0', linestyle='--', alpha=0.6)
plt.axvline(best_thr_stack, color='C1', linestyle='--', alpha=0.6)
plt.xlabel('Threshold')
plt.ylabel('F1 OOF')
plt.title('F1 vs Threshold — Averaging vs Stacking+calibration')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Soumission — meilleure stratégie automatique

In [ ]:
if best_f1_stack >= best_f1_avg:
    best_strategy  = 'Stacking+calibration'
    y_pred_proba   = test_stack
    best_threshold = best_thr_stack
    best_f1        = best_f1_stack
else:
    best_strategy  = 'Averaging'
    y_pred_proba   = test_avg
    best_threshold = best_thr_avg
    best_f1        = best_f1_avg

print(f'Stratégie choisie : {best_strategy} (F1 OOF={best_f1:.4f}, thr={best_threshold:.2f})')

y_pred_test = (y_pred_proba > best_threshold).astype(int)

submission = pd.DataFrame({
    'SEQN': test_seqn_order.values,
    'prediction': y_pred_test
}).sort_values('SEQN')

assert len(submission) == 5000, f'ERREUR : {len(submission)} lignes'

filename = f'{GROUP_ID}_{SUBMISSION_ID}.csv'
submission.to_csv(filename, index=False, header=False)

print(f'Fichier : {filename}')
print(f'Lignes  : {len(submission)}')
print(f'Répartition : {submission["prediction"].value_counts().to_dict()}')